In [ ]:
# Step 5: Display the best parameters and results
print('Best parameters:', study.best_params)
print('Best AUC:', study.best_value)

In [ ]:
# Step 4: Run the Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5)

In [ ]:
# Step 3: Define the Optuna objective function
def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'is_unbalance': True,
        'num_leaves': trial.suggest_int('num_leaves', 25, 4000),
        'max_depth': trial.suggest_int('max_depth', 5, 63),
        'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 0.05),
        'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 0.05),
        'min_child_samples': trial.suggest_int('min_child_samples', 50, 10000),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 100, 2000),
        'learning_rate': 0.03,
        'subsample_freq': 5,
        'bagging_seed': 42,
        'verbosity': -1
    }
    model = LGBMClassifier(**params)
    auc = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc').mean()
    return auc

In [ ]:
# Step 2: Load and prepare data
train_df = pd.read_csv('data/flight_delays_train.csv')
test_df = pd.read_csv('data/flight_delays_test.csv')
y_train = train_df['dep_delayed_15min'].map({'Y': 1, 'N': 0}).values

def label_enc(df_column):
    from sklearn.preprocessing import LabelEncoder
    return LabelEncoder().fit_transform(df_column)

def feature_eng(df):
    df['flight'] = df['Origin']+df['Dest']
    df['Month'] = df.Month.map(lambda x: x.split('-')[-1]).astype('int32')
    df['DayofMonth'] = df.DayofMonth.map(lambda x: x.split('-')[-1]).astype('uint8')
    df['begin_of_month'] = (df['DayofMonth'] < 10).astype('uint8')
    df['midddle_of_month'] = ((df['DayofMonth'] >= 10)&(df['DayofMonth'] < 20)).astype('uint8')
    df['end_of_month'] = (df['DayofMonth'] >= 20).astype('uint8')
    df['DayOfWeek'] = df.DayOfWeek.map(lambda x: x.split('-')[-1]).astype('uint8')
    df['hour'] = df.DepTime.map(lambda x: x/100).astype('int32')
    df['morning'] = df['hour'].map(lambda x: 1 if (x <= 11)& (x >= 7) else 0).astype('uint8')
    df['day'] = df['hour'].map(lambda x: 1 if (x >= 12) & (x <= 18) else 0).astype('uint8')
    df['evening'] = df['hour'].map(lambda x: 1 if (x >= 19) & (x <= 23) else 0).astype('uint8')
    df['night'] = df['hour'].map(lambda x: 1 if (x >= 0) & (x <= 6) else 0).astype('int32')
    df['winter'] = df['Month'].map(lambda x: x in [12, 1, 2]).astype('int32')
    df['spring'] = df['Month'].map(lambda x: x in [3, 4, 5]).astype('int32')
    df['summer'] = df['Month'].map(lambda x: x in [6, 7, 8]).astype('int32')
    df['autumn'] = df['Month'].map(lambda x: x in [9, 10, 11]).astype('int32')
    df['holiday'] = (df['DayOfWeek'] >= 5).astype(int)
    df['weekday'] = (df['DayOfWeek'] < 5).astype(int)
    df['airport_dest_per_month'] = df.groupby(['Dest', 'Month'])['Dest'].transform('count')
    df['airport_origin_per_month'] = df.groupby(['Origin', 'Month'])['Origin'].transform('count')
    df['airport_dest_count'] = df.groupby(['Dest'])['Dest'].transform('count')
    df['airport_origin_count'] = df.groupby(['Origin'])['Origin'].transform('count')
    df['carrier_count'] = df.groupby(['UniqueCarrier'])['Dest'].transform('count')
    df['carrier_count_per month'] = df.groupby(['UniqueCarrier', 'Month'])['Dest'].transform('count')
    df['deptime_cos'] = df['DepTime'].map(lambda x: np.cos(x * 2 * np.pi / 2400))
    df['deptime_sin'] = df['DepTime'].map(lambda x: np.sin(x * 2 * np.pi / 2400))
    df['flightUC'] = df['flight']+df['UniqueCarrier']
    df['DestUC'] = df['Dest']+df['UniqueCarrier']
    df['OriginUC'] = df['Origin']+df['UniqueCarrier']
    return df.drop('DepTime', axis=1)

import numpy as np
full_df = pd.concat([train_df.drop('dep_delayed_15min', axis=1), test_df])
full_df = feature_eng(full_df)
for column in ['UniqueCarrier', 'Origin', 'Dest','flight',  'flightUC', 'DestUC', 'OriginUC']:
    full_df[column] = label_enc(full_df[column])
X_train = full_df[:train_df.shape[0]]
X_test = full_df[train_df.shape[0]:]
categorical_features = ['Month',  'DayOfWeek', 'UniqueCarrier', 'Origin', 'Dest','flight',  'flightUC', 'DestUC', 'OriginUC']

In [ ]:
# Step 1: Install and import required libraries
!pip install optuna lightgbm
import optuna
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score

# Bayesian Optimization with Optuna
This notebook demonstrates a modern workflow for hyperparameter optimization using Optuna and LightGBM for flight delay prediction.

## Outline of the New Workflow
1. Install and import required libraries
2. Load and prepare data
3. Define the Optuna objective function
4. Run the Optuna study
5. Display the best parameters and results